# Lucrarea de laborator nr. 1. Statistică descriptivă și primele ipoteze

## Contextul sarcinii

Sunteți analist de date la compania de asigurări **MedGuard**, care vinde polițe de asigurare medicală facultativă în patru regiuni ale SUA.

Product managerul (PM) pregătește revizuirea tarifelor și a strategiei de marketing pentru anul următor. Înainte de a discuta orice cu departamentul financiar, are nevoie de un **portret real al bazei de clienți** și de verificarea câtorva afirmații care, în prezent, sunt acceptate în companie „pe încredere”.

Din corespondența internă a PM-ului:

> „Avem un export cu 1 338 de asigurați pentru anul trecut. Trebuie să înțeleg cine sunt clienții noștri, cât cheltuim cu adevărat pe ei și dacă e adevărat ce spun marketingul și finanțele. Am nevoie de răspunsuri cu cifre, nu cu impresii. Fiecare concluzie — cu argumentul pentru care ești sigur de ea.”

**Sarcina voastră** — să parcurgeți datele de la statistica descriptivă până la verificarea ipotezelor și, la final, să scrieți un raport scurt pentru PM.

---

## Setul de date

Folosim setul de date deschis **Medical Cost Personal Dataset** (despăgubiri de asigurare pe clienți).

**Descărcați fișierul singuri** de la linkul de mai jos și puneți-l în același folder cu acest notebook:

- https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv (click dreapta → „Salvează ca…” → `insurance.csv`)
- Oglindă pe Kaggle: https://www.kaggle.com/datasets/mirichoi0218/insurance

Descrierea coloanelor:

| Coloană | Descriere |
|---|---|
| `age` | vârsta asiguratului, ani |
| `sex` | sexul: `male` / `female` |
| `bmi` | indicele de masă corporală (kg/m²) |
| `children` | numărul de copii acoperiți de poliță |
| `smoker` | dacă clientul fumează: `yes` / `no` |
| `region` | regiunea de reședință: `northeast`, `northwest`, `southeast`, `southwest` |
| `charges` | suma despăgubirilor de asigurare pe an, USD |

---

## Reguli de lucru

1. Toate calculele — în cod (`numpy`, `pandas`, `scipy.stats`, `matplotlib`/`seaborn`), ca în prelegere.
2. După fiecare sarcină — **concluzie în cuvinte** într-o celulă markdown: ce am calculat, ce înseamnă pentru business.
3. Nivelul de semnificație în toate verificările de ipoteze: $\alpha = 0.05$.
4. Pentru verificarea ipotezelor folosim **testul z pentru medie** și intervalele de încredere, ca în prelegere (eșantionul este mare, $N > 30$, deci în locul lui $\sigma$ al PS luăm $sd$ al eșantionului).
5. Celulele cu `# codul vostru` — le completați voi. Celulele cu `**Concluzie:**` — la fel.

In [ ]:
import math
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")
np.random.seed(42)  # pentru ca rezultatele cu aleatoriu să fie reproductibile

df = pd.read_csv("insurance.csv")
print(df.shape)
df.head()

## Sarcina 1. Cunoașterea datelor și tipurile de variabile

PM cere, pentru început, „pur și simplu să descriem ce tabel avem”. Dar înainte de a calcula orice, trebuie să înțelegem **cu ce tipuri de variabile** lucrăm — de asta depinde ce măsuri și grafice sunt admisibile.

**1.1.** Afișați informații generale despre tabel: numărul de rânduri și coloane, tipurile de date, prezența valorilor lipsă (`df.info()`, `df.isna().sum()`, `df.describe(include="all")`).

**1.2.** Completați tabelul de mai jos: pentru fiecare coloană determinați tipul variabilei conform clasificării din prelegere
(cantitativă continuă / cantitativă discretă / nominală / ordinală) și **argumentați într-o singură propoziție**.

**1.3.** Răspundeți: pentru care dintre coloane ar fi **o greșeală** să calculăm media, chiar dacă pandas permite acest lucru? De ce?

In [ ]:
# codul vostru: informații generale despre date


**Concluzie (completați tabelul):**

| Coloană | Tipul variabilei | Argument |
|---|---|---|
| `age` | | |
| `sex` | | |
| `bmi` | | |
| `children` | | |
| `smoker` | | |
| `region` | | |
| `charges` | | |

**Răspuns la 1.3:** 

## Sarcina 2. Populația statistică și eșantionul

Departamentul financiar propune să economisim: în loc să prelucrăm întreaga bază, să tragem concluzii pe baza unui **eșantion de 100 de clienți**. PM întreabă cât de sigur este acest lucru și cum se aleg corect acești 100 de oameni.

În cadrul sarcinii considerăm că tabelul nostru cu 1 338 de clienți este **populația statistică (PS)**.

**2.1.** Faceți un **eșantion aleator simplu** de 100 de clienți (`df.sample(n=100, random_state=...)`). Comparați media `charges` și proporția fumătorilor din eșantion cu cele din PS.

**2.2.** Faceți un **eșantion stratificat** de 100 de clienți, unde straturile sunt `region` (din fiecare regiune luăm proporțional cu ponderea ei în PS; indiciu: `df.groupby("region", group_keys=False).apply(lambda g: g.sample(frac=100/len(df), random_state=...))`). Comparați aceiași indicatori cu PS.

**2.3.** Repetați eșantionul aleator simplu de 5 ori cu `random_state` diferit și urmăriți cum „sare” media `charges`. Notați media minimă și maximă obținută.

**2.4.** Răspundeți PM-ului: putem lua decizii pe baza unui singur eșantion de 100 de clienți? Care dintre cele două metode de selecție ați recomanda-o și de ce? Ce alte straturi (în afară de regiune) ar avea sens să folosim în această sarcină?

In [ ]:
# codul vostru: eșantion aleator simplu și comparație cu PS


In [ ]:
# codul vostru: eșantion stratificat pe regiune și comparație cu PS


In [ ]:
# codul vostru: 5 repetări ale eșantionului aleator simplu, min și max al mediei charges


**Concluzie:** 

## Sarcina 3. Portretul clientului: măsuri ale tendinței centrale

PM vrea o singură cifră pentru slide: „clientul mediu MedGuard este …”.

**3.1.** Pentru variabilele `age`, `bmi`, `charges` calculați **modul, mediana și media** (`stats.mode`, `np.median`, `np.mean` sau metodele pandas). Adunați rezultatul într-un singur tabel.

**3.2.** Construiți histogramele acestor trei variabile (câte una pentru fiecare, `bins=30`). Pe fiecare histogramă marcați cu linii verticale media și mediana.

**3.3.** Pentru fiecare variabilă răspundeți: distribuția este simetrică sau asimetrică? Există valori aberante vizibile / mai multe moduri? Ce măsură a tendinței centrale **recomandați PM-ului pentru slide** și de ce (bazați-vă pe secțiunea „Exemple de utilizare” din prelegere)?

**3.4.** Verificați proprietatea mediei: PM vrea să prezinte despăgubirile **în mii de dolari**. Creați coloana `charges_k = charges / 1000` și arătați că media s-a schimbat exact de 1000 de ori, **fără a recalcula media din date** — doar prin proprietatea $M_{x \cdot c} = c \cdot M_x$.

In [ ]:
# codul vostru: mod, mediană, medie pentru age, bmi, charges (un singur tabel)


In [ ]:
# codul vostru: histograme cu liniile mediei și medianei


In [ ]:
# codul vostru: proprietatea mediei la înmulțirea cu o constantă


**Concluzie (ce măsură și de ce pentru fiecare variabilă):** 

## Sarcina 4. Cât de diferiți sunt clienții: măsuri ale variabilității

Departamentul financiar afirmă: „Fumătorii sunt un segment previzibil, sunt pur și simplu mai scumpi”. PM are îndoieli: mai scumpi — da, dar **mai previzibili**?

**4.1.** Împărțiți clienții în două grupuri după `smoker`. Pentru `charges` în fiecare grup calculați: amplitudinea ($R$), dispersia ($D$, `ddof=1`), abaterea standard ($sd$, `ddof=1`). Nu uitați de corecția $n-1$ — explicați într-o propoziție de ce este necesară.

**4.2.** Calculați aceiași indicatori **manual**, după formulele din prelegere, pentru grupul fumătorilor (printr-un ciclu sau vectorial: abaterile de la medie → pătrate → sumă → împărțire la $n-1$ → rădăcină) și convingeți-vă că rezultatul coincide cu `np.var` / `np.std`.

**4.3.** Proprietatea dispersiei. Anul viitor toate despăgubirile vor crește cu o valoare fixă de 500 USD (indexare). Cum se vor schimba $D$ și $sd$ pentru `charges`? Dar dacă despăgubirile cresc cu 10% (înmulțire cu 1.1)? Răspundeți mai întâi **prin proprietăți**, apoi verificați cu cod.

**4.4.** Răspundeți PM-ului: care segment este mai omogen ca despăgubiri — fumătorii sau nefumătorii? Ce înseamnă asta pentru planificarea bugetului?

In [ ]:
# codul vostru: amplitudine, dispersie, sd pe grupuri smoker


In [ ]:
# codul vostru: calcul manual al dispersiei și sd pentru fumători


In [ ]:
# codul vostru: proprietățile dispersiei (+500 și *1.1)


**Concluzie:** 

## Sarcina 5. Clienți anormal de scumpi: cuartile, IQR, box plot

PM: „Am impresia că vreo câțiva clienți «mănâncă» tot bugetul. Câți sunt de fapt și cine sunt?”

**5.1.** Pentru `charges` calculați $Q1$, $Q2$ (mediana), $Q3$ și $IQR$ (`np.percentile` sau `df.charges.quantile`).

**5.2.** După algoritmul din prelegere calculați limitele valorilor aberante: $Q1 - 1.5 \cdot IQR$ și $Q3 + 1.5 \cdot IQR$. Calculați **câți clienți** sunt valori aberante și **ce pondere** din totalul despăgubirilor companiei reprezintă ei.

**5.3.** Construiți box plot pentru `charges` pe grupuri `smoker` (`sns.boxplot(x="smoker", y="charges", data=df)`), apoi pe `region`. Descrieți ce se vede.

**5.4.** Priviți „valorile aberante” de la p. 5.2: ce proporție de fumători există printre ele? Ce `bmi` mediu au? Comparați cu baza în ansamblu.

**5.5.** Răspundeți PM-ului: sunt acestea într-adevăr anomalii (erori de date care trebuie șterse) sau un **segment firesc**? Ce recomandați: să le eliminăm din analiză, să le analizăm separat sau să le lăsăm așa cum sunt? Argumentați.

In [ ]:
# codul vostru: cuartile, IQR, limitele valorilor aberante, numărul lor și ponderea despăgubirilor


In [ ]:
# codul vostru: box plot pe smoker și pe region


In [ ]:
# codul vostru: portretul clienților-valori aberante (proporția fumătorilor, bmi mediu)


**Concluzie:** 

## Sarcina 6. Distribuția normală, scorurile z și regula 2–3 sigma

Departamentul de underwriting vrea să introducă o **suprataxă pentru BMI ridicat**: „clienții cu BMI peste 40 sunt o raritate, pot fi evaluați individual”. Trebuie verificat cât de rari sunt.

**6.1.** Construiți histograma `bmi` și suprapuneți curba densității distribuției normale cu parametrii $\mu = \bar{X}_{bmi}$, $\sigma = sd_{bmi}$ (`stats.norm.pdf`). Seamănă distribuția cu una normală „din ochi”?

**6.2.** Efectuați **standardizarea z** a coloanei `bmi` (creați coloana `bmi_z`). Convingeți-vă că noua medie este $\approx 0$, iar abaterea standard $\approx 1$. S-a schimbat forma histogramei?

**6.3.** Clientul cu ID = 1317 (ultimul rând al tabelului, `df.iloc[-1]`) — care este scorul lui z pentru `bmi`? Cu câte abateri standard diferă de clientul mediu și în ce direcție?

**6.4.** Folosind modelul normal ($\mu$, $\sigma$ de la p. 6.1) și `stats.norm.sf` / `.cdf`, calculați proporția **teoretică** a clienților cu `bmi > 40`. Apoi calculați proporția **reală** din date. Cât de mult diferă?

**6.5.** După regula 2–3 sigma calculați câți clienți ar trebui să se afle în intervalul $[\mu - 2\sigma; \mu + 2\sigma]$ (se așteaptă $\approx 95.4\%$) și comparați cu realitatea.

**6.6.** Răspundeți departamentului de underwriting: este corectă afirmația „BMI > 40 este o raritate”? Ce prag ați propune dacă „raritate” înseamnă primii 5% dintre clienți (indiciu: `stats.norm.isf(0.05, mu, sigma)` și `df.bmi.quantile(0.95)` — comparați ambele variante).

In [ ]:
# codul vostru: histograma bmi + curba normală


In [ ]:
# codul vostru: standardizarea z a bmi


In [ ]:
# codul vostru: scorul z al clientului df.iloc[-1]


In [ ]:
# codul vostru: proporția teoretică și reală bmi > 40; regula 2 sigma; pragul 5%


**Concluzie:** 

## Sarcina 7. Teorema limită centrală pe datele noastre

Distribuția `charges` (sarcina 3) clar nu este normală. PM întreabă: „Atunci cum aveți de gând să vă încredeți în media unui eșantion?” Răspunsul este TLC. Arătați-o în practică.

Considerăm din nou tabelul cu 1 338 de clienți drept **populație statistică**.

**7.1.** Extrageți din PS **1000 de eșantioane** de câte $n = 50$ clienți (cu revenire, `df.charges.sample(n=50, replace=True)` într-un ciclu) și pentru fiecare calculați media `charges`. Construiți histograma celor 1000 de medii de eșantion.

**7.2.** Comparați:
- media PS și media mediilor de eșantion;
- abaterea standard a mediilor de eșantion (aceasta este $se$ empirică) și $se$ teoretică $= \dfrac{sd_{PS}}{\sqrt{n}}$.

**7.3.** Repetați p. 7.1 pentru $n = 10$ și $n = 200$. Construiți trei histograme alăturate (aceeași axă X). Cum se schimbă forma și lățimea distribuției mediilor de eșantion?

**7.4.** Răspundeți PM-ului în 2–3 propoziții: de ce putem folosi media eșantionului și intervalele de încredere chiar dacă despăgubirile în sine sunt distribuite puternic asimetric? De ce depinde precizia?

In [ ]:
# codul vostru: 1000 de eșantioane de câte 50, histograma mediilor de eșantion


In [ ]:
# codul vostru: compararea mediei PS cu media mediilor; se empirică vs teoretică


In [ ]:
# codul vostru: n = 10, 50, 200 — trei histograme


**Concluzie:** 

## Sarcina 8. Intervale de încredere pentru buget

Departamentul financiar planifică rezerva pentru despăgubiri. Are nevoie nu de o singură cifră, ci de un **interval** în care, cu încredere mare, se află despăgubirea medie pe client.

**8.1.** Luați un eșantion aleator de $n = 100$ clienți (`random_state=1`). Calculați $\bar{X}$, $sd$, $se = sd / \sqrt{n}$ pentru `charges`.

**8.2.** Construiți intervalele de încredere de **95%** și **99%** pentru media `charges`. Coeficienții 1.96 și 2.58 obțineți-i prin `stats.norm.isf`, ca în prelegere, nu „din memorie”.

**8.3.** Verificați: media **adevărată** a PS (pe toți cei 1 338 de clienți) cade în intervalele voastre?

**8.4.** Departamentul financiar este dispus să plătească pentru colectarea datelor de la încă 300 de clienți (în total $n = 400$). Fără a recalcula nimic din date, doar prin formula $se$, spuneți: de câte ori se va îngusta IÎ de 95%? Apoi verificați pe un eșantion real de 400 de clienți.

**8.5.** Construiți IÎ de 95% pentru media `charges` **separat pentru fumători și nefumători** (pe toată baza). Se intersectează intervalele? Ce îi spune asta PM-ului încă înainte de orice test formal?

**8.6.** Scrieți o frază pentru departamentul financiar în formatul: „Cu o încredere de 95%, despăgubirea medie pe client se află în intervalul de la … până la … USD” și explicați într-o propoziție ce înseamnă exact „95%” (amintiți-vă întrebarea din prelegere despre interpretarea corectă a IÎ).

In [ ]:
# codul vostru: eșantion n=100, X_, sd, se


In [ ]:
# codul vostru: IÎ 95% și 99%; verificarea includerii mediei PS


In [ ]:
# codul vostru: efectul creșterii n la 400


In [ ]:
# codul vostru: IÎ pentru fumători și nefumători


**Concluzie:** 

## Sarcina 9. Verificarea ipotezelor: ce este adevărat din „lucrurile știute” în companie?

Aceasta este partea principală a lucrării. În companie circulă patru afirmații; PM vrea pentru fiecare un verdict clar: **confirmăm / respingem**, cu valoarea $p$ și cu interval de încredere.

Pentru fiecare ipoteză formulați răspunsul după o schemă unică (ca în prelegere, secțiunea „Ideea inferenței statistice”):

1. Formulați $H_0$ și $H_1$ (alternativă bilaterală sau unilaterală — argumentați).
2. Calculați $\bar{X}$, $sd$, $n$, $se = sd/\sqrt{n}$.
3. Calculați $z = \dfrac{\bar{X} - M_0}{se}$.
4. Calculați valoarea $p$ (`stats.norm.cdf` / `.sf`; puteți folosi funcția `p_value` din prelegere — este mai jos).
5. Construiți IÎ de 95% și verificați dacă $M_0$ cade în el — rezultatele celor două abordări trebuie să concorde.
6. Verdict la $\alpha = 0.05$ și **concluzie pentru business** în una-două propoziții.

---

### Ipoteza A. „Un fumător mediu ne costă 30 000 USD pe an”
Marketingul folosește această cifră în prezentări de doi ani. Verificați: despăgubirea medie pentru fumători este egală cu $M_0 = 30\,000$ USD? (Alternativa este bilaterală: ne interesează orice abatere.)

### Ipoteza B. „Publicul nostru țintă sunt clienții de 39 de ani”
Strategia de produs este construită în jurul „clientului mediu de 39 de ani”. Verificați: vârsta medie a clienților este egală cu $M_0 = 39$ de ani?

### Ipoteza C. „Nord-estul se încadrează în bugetul de 12 500 USD pe client”
Finanțele au prevăzut în bugetul regiunii `northeast` o despăgubire medie de $M_0 = 12\,500$ USD. Verificați dacă există motive să considerăm că despăgubirea medie reală pentru această regiune diferă de buget.

### Ipoteza D. „Sud-estul este o regiune problematică, acolo despăgubirile depășesc 13 000 USD”
Managerul regiunii `southeast` cere majorarea bugetului, afirmând că despăgubirea medie de acolo **depășește** $M_0 = 13\,000$ USD. Aici alternativa este **unilaterală** („mai mare”) — explicați de ce acest lucru schimbă calculul valorii $p$.

### Ipoteza E (bonus, a voastră).
Formulați **propria** afirmație despre baza de clienți, care ar fi utilă PM-ului (de exemplu, despre `bmi`, `children` sau altă regiune), și verificați-o după aceeași schemă.

In [ ]:
# funcția din prelegere pentru calculul valorii p pe baza statisticii z
def p_value(z_stat, alternative="two-sided"):
    if alternative == "two-sided":
        return 2 * (1 - stats.norm.cdf(np.abs(z_stat)))
    if alternative == "less":
        return stats.norm.cdf(z_stat)
    if alternative == "greater":
        return 1 - stats.norm.cdf(z_stat)


def z_test_mean(sample, m0, alternative="two-sided", conf=0.95):
    '''
    Schiță: returnați un dicționar cu n, mean, sd, se, z, p, ci_low, ci_high.
    Completați singuri corpul funcției — și folosiți-o pentru toate ipotezele.
    '''
    # codul vostru
    pass

### Ipoteza A — „fumătorul mediu = 30 000 USD”

- $H_0$:
- $H_1$: 

In [ ]:
# codul vostru: ipoteza A


**Verdict și concluzie pentru business:** 

### Ipoteza B — „vârsta medie a clientului = 39 de ani”

- $H_0$:
- $H_1$: 

In [ ]:
# codul vostru: ipoteza B


**Verdict și concluzie pentru business:** 

### Ipoteza C — „northeast: despăgubirea medie = 12 500 USD”

- $H_0$:
- $H_1$: 

In [ ]:
# codul vostru: ipoteza C


**Verdict și concluzie pentru business:** 

### Ipoteza D — „southeast: despăgubirea medie > 13 000 USD”

- $H_0$:
- $H_1$:
- De ce alternativă unilaterală și cum influențează $p$: 

In [ ]:
# codul vostru: ipoteza D


**Verdict și concluzie pentru business:**